# 04 · Demucs: 음악 오디오 → 6 스템

**Hybrid Transformer Demucs** (Rouard, Massa & Défossez, ICASSP 2023). 파형과 스펙트로그램 두 갈래 U-Net 사이에 Transformer를 둔 스템 분리 모델. `htdemucs_6s`는 vocals · drums · bass · guitar · piano · other 6개로 나눕니다.

- 입력 → 출력: 음악 오디오 → 6개의 오디오
- 소요 시간: 설치 2분 + 60초 곡 분리 T4 약 20초 (CPU 약 2분)
- 논문: https://arxiv.org/abs/2211.08553

In [ ]:
#@title 0. 준비: 수업 자료 받기와 모델 폴더 정하기 (실행만 하면 됩니다)
#@markdown 모델 가중치는 모두 `models` 폴더 하나에 받습니다. 기본 위치는 `/content`이고 런타임을 끄면 지워집니다. 체크하면 Google Drive의 `CT-AI_Microdegree/` 아래에 `models`와 `outputs`를 두어 다음 실행 때 다시 받지 않고, 다른 노트북의 결과도 이어서 쓸 수 있습니다.
KEEP_IN_DRIVE = False  #@param {type:"boolean"}
import os, sys, glob, time, json, subprocess
_cands = [os.environ.get('CTAI_REPO_DIR'), os.path.abspath(os.path.join(os.getcwd(), '..', '..')), '/content/CT-AI_Microdegree']
REPO = next((c for c in _cands if c and os.path.isfile(os.path.join(c, 'colab', 'ctai.py'))), None)
if REPO is None:
    REPO = '/content/CT-AI_Microdegree'
    subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/dabinkim0/CT-AI_Microdegree', REPO], check=True)
sys.path.insert(0, os.path.join(REPO, 'colab'))
REQ = os.path.join(REPO, 'colab', 'requirements')   # 노트북별 pip 패키지 목록
import ctai
P = ctai.setup(keep_in_drive=KEEP_IN_DRIVE)
AUDIO_DIR, ASSET_DIR, MODELS_DIR, CODE_DIR, OUT_DIR, IN_COLAB = P['AUDIO_DIR'], P['ASSET_DIR'], P['MODELS_DIR'], P['CODE_DIR'], P['OUT_DIR'], P['IN_COLAB']
print('예시 음원:', sorted(os.path.basename(p) for p in glob.glob(os.path.join(AUDIO_DIR, '*.wav'))))


In [ ]:
#@title 설치
%pip install -q -r {REQ}/04_demucs.txt
import demucs, torch; print('설치 OK · demucs', demucs.__version__, '· torch', torch.__version__, '· GPU:', torch.cuda.is_available())

In [ ]:
#@title 오디오 준비: 예시 음원 / 업로드 / YouTube 중 선택
import ipywidgets as W, soundfile as sf, numpy as np, librosa
from IPython.display import Audio, display
import matplotlib.pyplot as plt

EXAMPLES = {'밴드 (Let\'s Go Fishin\', 보컬+밴드)': 'band_lets_go_fishin_60s.wav', '재즈 (Vibe Ace)': 'jazz_vibe_ace_60s.wav'}
SOURCE = '예시'      #@param ["예시", "업로드", "YouTube"]
EXAMPLE = "밴드 (Let's Go Fishin', 보컬+밴드)"   #@param ["밴드 (Let's Go Fishin', 보컬+밴드)", "재즈 (Vibe Ace)"]
YOUTUBE_URL = ''   #@param {type:"string"}
MAX_SECONDS = 60   #@param {type:"integer"}
TARGET_SR = 44100

def load_audio(path, sr=TARGET_SR, max_sec=MAX_SECONDS, mono=False):
    y, _ = librosa.load(path, sr=sr, mono=mono)
    if max_sec: y = y[..., :int(max_sec * sr)]
    return y, sr

if SOURCE == '예시':
    audio_path = os.path.join(AUDIO_DIR, EXAMPLES[EXAMPLE])
elif SOURCE == '업로드':
    from google.colab import files
    up = files.upload(); audio_path = list(up.keys())[0]
else:
    # 수업 내 학습 목적으로만 사용하세요. 저작권이 있는 음원은 외부에 배포하지 않습니다.
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'yt-dlp'], check=True)
    subprocess.run(['yt-dlp', '-q', '-x', '--audio-format', 'wav', '-o', 'yt_input.%(ext)s', YOUTUBE_URL], check=True)
    audio_path = 'yt_input.wav'

y, sr = load_audio(audio_path)
mono = y if y.ndim == 1 else y.mean(0)
print('파일:', audio_path, '| 길이: %.1f s | sr=%d' % (len(mono) / sr, sr))
plt.figure(figsize=(14, 2)); plt.plot(np.arange(len(mono)) / sr, mono, lw=.4, color='#58c4dd'); plt.xlabel('time (s)'); plt.title('waveform'); plt.tight_layout(); plt.show()
display(Audio(mono, rate=sr))


In [ ]:
#@title 모델 로드
from demucs.pretrained import get_model
from demucs.apply import apply_model
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = get_model('htdemucs_6s').to(device).eval()
print('스템:', model.sources, '| 파라미터 수: %.1f M' % (sum(p.numel() for p in model.parameters()) / 1e6), '| device:', device)

In [ ]:
#@title 추론: 분리 실행
wav = torch.tensor(y if y.ndim == 2 else np.stack([y, y]), dtype=torch.float32)   # (2, T)
ref = wav.mean(0); wav_n = (wav - ref.mean()) / (ref.std() + 1e-8)
t0 = time.time()
with torch.no_grad():
    out = apply_model(model, wav_n[None].to(device), device=device, shifts=1, split=True, overlap=0.25, progress=False)[0]
out = out * (ref.std() + 1e-8) + ref.mean()
stems = {name: out[i].cpu().numpy() for i, name in enumerate(model.sources)}
dt = time.time() - t0; dur = wav.shape[1] / sr
print('처리 시간 %.1f s / 음원 %.1f s → 실시간의 %.1f배 속도' % (dt, dur, dur / dt))

In [ ]:
#@title 듣기·보기: 스템별 에너지와 재생
fps = 50; hop = sr // fps
fig, axes = plt.subplots(len(stems), 1, figsize=(14, 1.2 * len(stems) + 1), sharex=True)
colors = {'vocals': '#fc6255', 'drums': '#58c4dd', 'bass': '#c48a5a', 'guitar': '#83c167', 'piano': '#b19cd9', 'other': '#a9a9a9'}
for ax, (name, s) in zip(axes, stems.items()):
    m = s.mean(0); rms = librosa.feature.rms(y=m, frame_length=hop * 2, hop_length=hop)[0]; rms = rms / (rms.max() + 1e-9)
    ax.fill_between(np.arange(len(rms)) / fps, 0, rms, color=colors.get(name, '#888'), alpha=.8); ax.set_ylim(0, 1); ax.set_yticks([]); ax.set_ylabel(name, rotation=0, ha='right')
axes[-1].set_xlabel('time (s)'); plt.tight_layout(); plt.show()
print('원음'); display(Audio(wav.numpy(), rate=sr))
for name, s in stems.items():
    print(name); display(Audio(s, rate=sr))

## 해 보기

1. 아래 슬라이더로 스템을 섞어 **리믹스**를 만들어 보세요. 보컬만 빼면 MR(반주)이 됩니다.
2. 가장 분리가 **안 되는** 악기는 무엇인가요? (힌트: `other`와 `piano`를 혼자 들어 보기) 왜 그럴까요?

In [ ]:
#@title 리믹스: 스템 가중치 (0 = 끄기, 1 = 원래대로)
vocals = 1.0 #@param {type:"slider", min:0, max:1.5, step:0.1}
drums = 1.0  #@param {type:"slider", min:0, max:1.5, step:0.1}
bass = 1.0   #@param {type:"slider", min:0, max:1.5, step:0.1}
guitar = 1.0 #@param {type:"slider", min:0, max:1.5, step:0.1}
piano = 1.0  #@param {type:"slider", min:0, max:1.5, step:0.1}
other = 1.0  #@param {type:"slider", min:0, max:1.5, step:0.1}
w = dict(vocals=vocals, drums=drums, bass=bass, guitar=guitar, piano=piano, other=other)
remix = sum(w[k] * stems[k] for k in stems)
display(Audio(remix, rate=sr))

In [ ]:
#@title 저장: 스템 wav를 outputs/stems 폴더에
stem_dir = os.path.join(OUT_DIR, 'stems'); os.makedirs(stem_dir, exist_ok=True)
for name, s in stems.items(): sf.write(os.path.join(stem_dir, name + '.wav'), s.T, sr)
sf.write(os.path.join(stem_dir, 'vocals_20s_16k.wav'), librosa.resample(stems['vocals'].mean(0)[:20 * sr], orig_sr=sr, target_sr=16000), 16000)   # 05 Tone Transfer 입력용
print('저장:', sorted(os.listdir(stem_dir)))